# Particle Path Notebook
This notebook is for simulating the trajectory of charged particles through an ion thruster. [Magpylib](https://magpylib.readthedocs.io) is used for defining the static magnetic field of the thruster geometry. [PlasmaPy](https://docs.plasmapy.org) is used for running the charged particle simulation.

### Import Libraries
Along with magpylib and PlasmaPy, [astropy.units](https://docs.astropy.org/en/stable/units/index.html) is used for all physical units and numpy is used for arrays. [PyVista](https://pyvista.org/) will be used for visualization.

In [1]:
import astropy.units as u
import numpy as np

import magpylib as magpy
from plasmapy.particles import Particle
from plasmapy.plasma.grids import CartesianGrid
from plasmapy.simulation.particle_tracker.particle_tracker import ParticleTracker
from plasmapy.simulation.particle_tracker.save_routines import IntervalSaveRoutine
from plasmapy.simulation.particle_tracker.termination_conditions import (
    TimeElapsedTerminationCondition,
)

import pyvista as pv

## Thruster Magnet Geometry

### Define The Grid™
This `CartesianGrid` object stores a discrete cache of the magnetic (B) field in a grid of vertices. Using this, the simulation does not need to recalculate the field before each step, and can instead interpolate from the grid of precalculated points. The parameters for `CartesianGrid` below define the physical dimensions of the grid along with the number of points. If `num` is 10, the grid will be a 10x10x10 cube of 1000 points.

In [2]:
grid = CartesianGrid(-0.1 * u.m, 0.1 * u.m, num=50)

### Create Some Magnets
This next section will define the positions and polarities of each magnet. We use a nested for-loop to iterate over a radial and linear sequence.

In [19]:
RADIUS = 0.02
LAYER_SEPARATION = 0.03
POLARIZATION = 0.1
MAGNET_SIZE = (0.01, 0.01, 0.01)

RADIAL_COUNT = 4
LAYER_COUNT = 3

template_magnet = magpy.magnet.Cuboid(dimension=(0.01, 0.01, 0.01), style_opacity=0.3)
magnet_collection = magpy.Collection()

for layer_index in range(LAYER_COUNT):
    for radial_index in range(RADIAL_COUNT):
        array_index = layer_index * RADIAL_COUNT + radial_index

        magnet = template_magnet.copy()

        # The position at array_index is (x, y, z):
        theta = radial_index * 2 * np.pi / RADIAL_COUNT
        magnet.position = (
            layer_index * LAYER_SEPARATION - (LAYER_SEPARATION * LAYER_COUNT / 2),
            np.sin(theta) * RADIUS,
            np.cos(theta) * RADIUS,
        )

        # The polarization is negative if layer_index is odd
        magnet.polarization = (
            POLARIZATION if layer_index % 2 == 0 else -POLARIZATION,
            0.0,
            0.0,
        )

        magnet_collection.add(magnet)

### Put the B field in The Grid™
Update the grid with vectors calculated from the magnets.

In [4]:
B = magnet_collection.getB(grid.grid) * u.T

grid.add_quantities(B_x=B[:, :, :, 0], B_y=B[:, :, :, 1], B_z=B[:, :, :, 2])

## Particle Simulation
### Simulation Setup
First, we set up some initial conditions. `x0` defines the initial position and `v0` defines the initial velocity.

In [5]:
x0 = np.array([[-0.070, 0.002, 0]], dtype=np.float32) * u.m
v0 = np.array([[2000, 100, 0]], dtype=np.float32) * u.m / u.s
particle = Particle("p+")

termination_condition = TimeElapsedTerminationCondition(0.00006 * u.second)
save_routine = IntervalSaveRoutine(0.0000001 * u.second)

The termination condition and save routine define how long and how many frames we will get out of the simulation.

### Running the Simulation
At last, we create and run the simulation and obtain results.

In [ ]:
simulation = ParticleTracker(
    grid,
    save_routine=save_routine,
    termination_condition=termination_condition,
    verbose=False,
)

simulation.load_particles(x0, v0, particle)
simulation.run()

particle_trajectory = save_routine.results["x"][:, 0]

d:\Programming\Python\MagnaPy\env\Lib\site-packages\plasmapy\simulation\particle_tracker\particle_tracker.py:584: RuntimeWarning: Quantities should go to zero at edges of grid to avoid non-physical effects, but a value of 2.11E-04 T was found on the edge of the B_x array of grid 0. Consider applying a envelope function to force the quantities at the edge to go to zero.
  warnings.warn(
d:\Programming\Python\MagnaPy\env\Lib\site-packages\plasmapy\simulation\particle_tracker\particle_tracker.py:584: RuntimeWarning: Quantities should go to zero at edges of grid to avoid non-physical effects, but a value of 9.77E-05 T was found on the edge of the B_y array of grid 0. Consider applying a envelope function to force the quantities at the edge to go to zero.
  warnings.warn(
d:\Programming\Python\MagnaPy\env\Lib\site-packages\plasmapy\simulation\particle_tracker\particle_tracker.py:584: RuntimeWarning: Quantities should go to zero at edges of grid to avoid non-physical effects, but a value of 

<Quantity [[-6.7961730e-02,  2.1017499e-03, -5.3100816e-06],
           [-6.5923445e-02,  2.2031311e-03, -1.2545641e-05],
           [-6.3885137e-02,  2.3043249e-03, -1.4606206e-05],
           [-6.1846886e-02,  2.4058488e-03, -2.2561662e-06],
           [-5.9809070e-02,  2.5079334e-03,  4.0286683e-05],
           [-5.7772845e-02,  2.6087170e-03,  1.3281130e-04],
           [-5.5741016e-02,  2.6989949e-03,  3.0150442e-04],
           [-5.3719256e-02,  2.7555239e-03,  5.7394657e-04],
           [-5.1715314e-02,  2.7322073e-03,  9.5944019e-04],
           [-4.9737673e-02,  2.5614526e-03,  1.4334728e-03],
           [-4.7786307e-02,  2.1896316e-03,  1.9012862e-03],
           [-4.5853365e-02,  1.6200085e-03,  2.2240775e-03],
           [-4.3927591e-02,  9.4758411e-04,  2.2886870e-03],
           [-4.1996177e-02,  3.2764120e-04,  2.0645300e-03],
           [-4.0054832e-02, -1.1571466e-04,  1.6178268e-03],
           [-3.8104650e-02, -3.3858407e-04,  1.0592140e-03],
           [-3.6148570e-

## Viewing the Results
Lastly, we need to put the results in something easy to view. Using PyVista, we can plot the magnets and the particle trajectory.

In [ ]:
plotter = pv.Plotter()

magpy.show(magnet_collection, canvas=plotter)
plotter.add_lines(particle_trajectory * 1000, color="blue", width=2, connected=True)

plotter.show(jupyter_backend="client")

trigger(trigger__29)
trigger(trigger__30)
js_key = class
js_key = style
js_key = fluid
js_key = class
before: class = { 'rounded-circle': !P_0x25e8b42b910_9_show_ui }
(prefix=None) token {
has({ => {) = False
(prefix=None) token  
has(  =>  ) = False
(prefix=None) token '
has(' => ') = False
(prefix=None) token rounded
has(rounded => rounded) = False
(prefix=None) token -
has(- => -) = False
(prefix=None) token circle
has(circle => circle) = False
(prefix=None) token '
has(' => ') = False
(prefix=None) token :
has(: => :) = False
(prefix=None) token  
has(  =>  ) = False
(prefix=None) token !
has(! => !) = False
(prefix=None) token P_0x25e8b42b910_9_show_ui
has(P_0x25e8b42b910_9_show_ui => P_0x25e8b42b910_9_show_ui) = True
(prefix=None) translated P_0x25e8b42b910_9_show_ui
(prefix=None) token  
has(  =>  ) = False
(prefix=None) token }
has(} => }) = False
 => { 'rounded-circle': !P_0x25e8b42b910_9_show_ui }
after: class = { 'rounded-circle': !P_0x25e8b42b910_9_show_ui }
js_key = style


Widget(value='<iframe src="http://localhost:58539/index.html?ui=P_0x25e8b42b910_9&reconnect=auto" class="pyvis…